# AWS Glue Fundamentals & Local Development Simulation
This notebook teaches the core concepts of AWS Glue for serverless ETL (Extract, Transform, Load) pipelines and metadata cataloging.

## Objectives
1. **Understand AWS Glue Architecture**: GlueContext, SparkContext, and Job Bookmarks.
2. **Master DynamicFrames**: How they compare to PySpark DataFrames.
3. **Execute Core Transformations**: `resolveChoice`, `dropNullFields`, `unnest`, and `selectFields`.
4. **Local Development Integration**: Test Glue ETL jobs locally using PySpark wrapper models.

---

### Glue vs. Spark: Architectural Hierarchy
In AWS Glue, the ETL runtime sits on top of Apache Spark:
- **SparkContext (sc)**: Entry point for Spark functionality; coordinates jobs, tasks, and executors.
- **GlueContext**: Extends `SparkContext` to provide AWS-specific integrations (S3 connectors, Data Catalog tables, Data Quality rules).
- **Job**: Tracks execution state, configures job bookmarks, and handles transaction commits.

```
       +------------------------------------+
       |          AWS Glue Job              |
       |  (State Tracking & Job Bookmarks)  |
       +------------------------------------+
                         |
       +------------------------------------+
       |          GlueContext               |
       |  (S3 Readers, Catalog, DQ Rules)   |
       +------------------------------------+
                         |
       +------------------------------------+
       |          SparkContext              |
       |     (RDDs, Cluster Management)     |
       +------------------------------------+
```


In [ ]:
import sys
import os

# Set system path to import src modules
sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath("."))))

from src.glue_context import get_glue_context, get_job
from src.s3_integration import S3DataConnector
from src.dynamic_frame_ops import resolve_choices, drop_null_fields, select_fields, unnest_nested_fields
from src.data_quality_glue import GlueDataQualityEvaluator

# Initialize the Mock/Native context
sc, glue_context, spark_session = get_glue_context()
job = get_job(glue_context)

print(f"Spark Version: {spark_session.version}")
print(f"Glue Context Class: {glue_context.__class__.__name__}")


## DynamicFrame vs. Spark DataFrame: Core Differences

| Feature | AWS Glue DynamicFrame | Apache Spark DataFrame |
| :--- | :--- | :--- |
| **Schema Ingestion** | Dynamic (computes/evaluates types per record) | Static (requires parsing full dataset or setting schema) |
| **Heterogeneous Types**| Supports multiple active types in a column via `ChoiceType` | Enforces a single type per column (others cast or nullified) |
| **S3 Partitioning** | Automated indexing of hive partitioning during execution | Manual path listing / schema inference on directories |
| **Nested Data** | Optimized operations (`unnest()`) without complex SQL select exprs | Requires structured selection (`col.sub_col`) |
| **Error Handling** | Routes bad records to error folders without failing the execution | Job fails immediately or nullifies bad values on parse errors |


In [ ]:
# Initialize the connector
connector = S3DataConnector(glue_context)

# Load CSV data (mapped to local sample csv in mock mode)
sales_dyf = connector.read_from_s3(
    s3_path="s3://data-engineering-diaries-glue-demo/raw/sales/",
    file_format="csv"
)

print(f"Record Count: {sales_dyf.count()}")
print("\n--- Original Schema ---")
sales_dyf.printSchema()

print("\n--- First 3 Records ---")
sales_dyf.show(3)


## DynamicFrame Transformations
We will perform standard transforms:
1. **`resolve_choices`**: Cast `revenue` and `discount` columns to `double` types.
2. **`select_fields`**: Prune the dataset down to core analytical fields.
3. **`drop_null_fields`**: Clean out columns that are entirely null.


In [ ]:
# 1. Cast column types (resolveChoice specs)
schema_specs = [
    ("revenue", "cast:double"),
    ("discount", "cast:double")
]
transformed_dyf = resolve_choices(sales_dyf, specs=schema_specs)

# 2. Select specific fields
selected_fields = ["order_id", "customer_id", "order_date", "revenue", "discount", "status"]
transformed_dyf = select_fields(transformed_dyf, fields=selected_fields)

# 3. Drop null fields
transformed_dyf = drop_null_fields(transformed_dyf)

print("--- Transformed Schema ---")
transformed_dyf.printSchema()

print("\n--- Transformed Data Sample ---")
transformed_dyf.show(5)


## AWS Glue Data Quality (DQDL) Rules
AWS Glue Data Quality allows us to write assertions about our data directly inside our ETL pipeline.
- If data quality fails, we can capture metrics, write reports to S3, or alert systems via Amazon SNS.
- Let's run a simulation of our Data Quality evaluator below to verify our rules.


In [ ]:
# Define DQDL rules
rules = [
    "RowCount > 0",
    "ColumnCount == 8",
    "ColumnExists \"order_id\"",
    "IsComplete \"order_id\"",
    "IsComplete \"customer_id\"",
    "ColumnDataType \"revenue\" = \"Double\""
]

# Evaluate rules
dq_evaluator = GlueDataQualityEvaluator("sales_quality_rules", rules)
passed, report, _ = dq_evaluator.evaluate(sales_dyf)

print(f"Data Quality Evaluation Passed? {passed}\n")
print("Detailed Rule Execution Report:")
for idx, r in enumerate(report, 1):
    print(f"{idx}. [{r['Status']}] Rule: {r['Rule']} -> (Actual: {r['ActualValue']})")
